# Telecom Network Fault Prediction and Predictive Maintenance System
## End-to-End Machine Learning, PyTorch Autoencoder Anomaly Detection & XAI Pipeline

This notebook demonstrates the complete data processing, supervised model training, PyTorch autoencoder anomaly detection, risk score fusion, and explainable AI (XAI) workflow on the **Telstra Recruiting Network Disruptions Dataset**.

In [ ]:
import os
import sys
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, roc_auc_score, precision_recall_curve, auc
from sklearn.preprocessing import StandardScaler

# Ensure project root is in sys.path
sys.path.insert(0, "..")
from model.autoencoder import TelecomDenseAutoencoder

### 1. Data Ingestion & Preprocessing

In [ ]:
master_df = pd.read_csv("../data/telecom_failure_master.csv")
print(f"Master Dataset Shape: {master_df.shape}")
master_df.head()

### 2. Exploratory Data Analysis (EDA)

In [ ]:
# Target Variable Distribution
plt.figure(figsize=(6, 4))
sns.countplot(data=master_df, x='fault_severity', palette='viridis')
plt.title("Fault Severity Class Distribution (0=Low, 1=Medium, 2=Severe)")
plt.show()

### 3. Supervised Classification Model Training

In [ ]:
feature_cols = joblib.load("../model/model_columns.pkl")
X = master_df[feature_cols]
y = master_df['fault_severity']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

clf = RandomForestClassifier(n_estimators=150, max_depth=15, random_state=42, class_weight='balanced')
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1 Macro:", f1_score(y_test, y_pred, average='macro'))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

### 4. PyTorch Autoencoder Anomaly Detection

In [ ]:
scaler = joblib.load("../model/autoencoder_scaler.pkl")
threshold = joblib.load("../model/autoencoder_threshold.pkl")

X_scaled = scaler.transform(X)
model = TelecomDenseAutoencoder(input_dim=len(feature_cols))
model.load_state_dict(torch.load("../model/autoencoder_model.pt"))
model.eval()

with torch.no_grad():
    tensor_x = torch.tensor(X_scaled, dtype=torch.float32)
    recon = model(tensor_x)
    mse_errors = torch.mean((tensor_x - recon) ** 2, dim=1).numpy()

y_binary = (y >= 1).astype(int)
print("Anomaly Threshold (95th Percentile):", threshold)
print("ROC-AUC Score:", roc_auc_score(y_binary, mse_errors))

### 5. Combined Operational Risk Fusion & XAI Attribution

In [ ]:
from ai.xai_explainer import TelecomRiskEngine
engine = TelecomRiskEngine(data_dir="../data", model_dir="../model")
sample_res = engine.analyze_ticket(10)
print("Ticket #10 Risk Category:", sample_res['risk_category'])
print("Combined Risk Score:", f"{sample_res['combined_risk']:.1%}")
print("Top Feature Attributions:", sample_res['top_attributions'][:3])